# CTNet-PD — NeuroVoz: experimentos consolidados

Este notebook reemplaza a `legacy/ctnet-neurovoz_original.ipynb`. Toda la lógica vive en el paquete `ctnet_pd` (`ctnet_pd/src/ctnet_pd`) y **todos los hiperparámetros** están en `ctnet_pd/configs/default.yaml`. El notebook solo orquesta, muestra y guarda resultados.

Cada sección indica a qué apartado de `paper/materials_methods_v2.md` corresponde.

### Qué cambió respecto al notebook legado

| Notebook legado | Problema | Ahora |
|---|---|---|
| `pad_audio`: recorte a media + 1,5·DE y "relleno" con `ifft(fft(x), n)` | La interpolación espectral **estira el audio** (cambia F0 y velocidad del habla) y descarta la parte imaginaria | Ventanas deslizantes de 229 frames; la última se rellena con silencio (§2.5) |
| `StratifiedKFold` sobre grabaciones | Un mismo hablante queda en train y test (fuga de información) | CV repetida, estratificada y **disjunta por sujeto** (§2.4) |
| El tuner y el `ModelCheckpoint` usan el fold de test como validación | Selección de modelo con datos de test | Validación interna por sujeto; el test solo se predice una vez |
| `ReduceLROnPlateau(min_lr=0.001)` con Adam a 1e-3 | La tasa de aprendizaje nunca bajaba | `min_lr = 1e-6`; la tasa se busca de verdad |
| Cabeza flatten → Dense (≈2,9 M parámetros con 14×25) | Sobreajuste con ~100 sujetos | GAP + lineal: 130 parámetros (§2.6) |
| Sin codificación posicional | La atención es invariante a permutaciones | Embedding posicional aprendido |
| `Saliency` presentado como "LRP" | Es gradiente simple, no LRP | Se reporta como *gradient saliency* (§2.9) |
| Grad-CAM con `layers[-4].activation = linear` | Modificaba una capa equivocada | Grad-CAM sobre `stem_out` y contribución exacta por token (§2.8) |
| `X[:42]` = HC, `X[42:]` = PD | `os.listdir` no ordena por clase | Los prototipos se calculan con las etiquetas (§2.12) |
| Métricas por grabación, F1/AUC sobre one-hot | Unidad estadística incorrecta | Métricas por **sujeto**, con IC por bootstrap de sujetos (§2.15) |
| Folds 1–4 cargados de una corrida previa, fold 5 reentrenado | Protocolos mezclados | Todos los folds bajo el mismo protocolo, reanudable |

## 0. Entorno (Kaggle)

In [ ]:
# Option A: clone the repository (needs internet enabled in the Kaggle session)
# !git clone --depth 1 -b claude/lucid-lovelace-bixixw https://github.com/AlejoPatigno/MIL.git /kaggle/working/MIL
# Option B: upload the ctnet_pd folder as a Kaggle Dataset (e.g. /kaggle/input/ctnet-pd)
!pip install -q keras-tuner

In [ ]:
import os, sys, json, platform
from pathlib import Path

CANDIDATES = [Path("/kaggle/working/MIL/ctnet_pd"), Path("/kaggle/input/ctnet-pd"), Path("../"), Path(".")]
PKG_ROOT = next(p.resolve() for p in CANDIDATES if (p / "src" / "ctnet_pd").exists())
sys.path.insert(0, str(PKG_ROOT / "src"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras, librosa, sklearn

# Let TensorFlow allocate GPU memory on demand, so the baseline extractors (PyTorch,
# run in a separate process) still find free GPU memory.
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

import ctnet_pd
from ctnet_pd.config import load_config, apply_overrides, token_grid
from ctnet_pd import data, features, splits, metrics, xai, faithfulness, sanity, reproducibility, analysis
from ctnet_pd.pipeline import VARIANTS, run_within_cohort, run_external, variant_config

VERSIONS = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
            "librosa": librosa.__version__, "scikit-learn": sklearn.__version__, "ctnet_pd": ctnet_pd.__version__,
            "gpus": [d.name for d in tf.config.list_physical_devices("GPU")]}
print(PKG_ROOT); print(VERSIONS)

## 1. Configuración (§2.5, §2.6, §2.14)

Todos los hiperparámetros salen de `configs/default.yaml`. Para cambiar uno, usa `OVERRIDES` en vez de editar el código, así cada corrida queda registrada.

In [ ]:
OUT = Path("/kaggle/working/outputs")
OVERRIDES = [
    "cohorts.neurovoz.root=/kaggle/input/neurovoz-dataset/data",
    # "cv.n_repeats=1",          # quick runs
    # "tuning.enabled=false",    # fixed hyperparameters from the YAML
]
cfg = load_config(PKG_ROOT / "configs" / "default.yaml", OVERRIDES)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "config_used.json").write_text(json.dumps({"config": cfg.to_dict(), "versions": VERSIONS}, indent=2))

def flatten(d, prefix=""):
    for k, v in d.items():
        if isinstance(v, dict):
            yield from flatten(v, f"{prefix}{k}.")
        else:
            yield f"{prefix}{k}", v

hp_table = pd.DataFrame(flatten(cfg.to_dict()), columns=["parameter", "value"])
print("token grid:", token_grid(cfg))
hp_table[hp_table.parameter.str.match(r"(audio|spectrogram|segmentation|model|training|cv|tuning)\.")]

## 2. Datos (§2.2, §2.3)

**Revisa la tabla de familias de tareas:** ninguna tarea de DDK ni de monólogo debe caer en `repeat` (la familia por defecto). Si alguna cae ahí, ajusta las expresiones regulares de `task_families` en el YAML.

In [ ]:
index = data.index_cohort(cfg, "neurovoz")
print(len(index), "recordings |", index.subject_id.nunique(), "subjects |",
      index.drop_duplicates("subject_id").group.value_counts().to_dict())
data.task_summary(index)

In [ ]:
print(sorted(index.task.unique()))

In [ ]:
meta = data.load_metadata(cfg, "neurovoz")
display(data.check_index_against_metadata(index, meta))
table1 = data.demographics_table(meta[meta.subject_id.isin(index.subject_id)])
display(table1)
print(data.demographic_tests(meta[meta.subject_id.isin(index.subject_id)]))
table1.to_csv(OUT / "table1_neurovoz.csv")

## 3. Log-Mel y segmentación (§2.5)

Los features se guardan en caché (`.npy` en float16 + `.csv`), así que esta celda solo tarda la primera vez.

In [ ]:
X, seg = features.build_features(index, cfg, cache_path=OUT / "cache" / "neurovoz_features")
print(X.shape, X.dtype)
seg.groupby("task_family").agg(segments=("fill", "size"), recordings=("recording_id", "nunique"),
                               mean_fill=("fill", "mean"), median_frames=("n_frames", "median"))

In [ ]:
rows, cols = token_grid(cfg)
i = int(np.flatnonzero(seg.task_family.eq("ddk_pataka").to_numpy())[0]) if seg.task_family.eq("ddk_pataka").any() else 0
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.imshow(np.asarray(X[i], dtype=np.float32), origin="lower", aspect="auto", cmap="magma")
from ctnet_pd.grid import tile_edges
for e in tile_edges(cfg.segmentation.frames, cols): ax.axvline(e - 0.5, color="w", lw=0.3, alpha=0.5)
for e in tile_edges(cfg.spectrogram.n_mels, rows): ax.axhline(e - 0.5, color="w", lw=0.3, alpha=0.5)
ax.set(title=f"{seg.filename[i]} (segment {seg.segment_idx[i]}) with the {rows}x{cols} token grid",
       xlabel="frame", ylabel="Mel bin")
plt.show()

## 4. Prueba rápida del pipeline

Corre 1 fold con pocas épocas y sin búsqueda de hiperparámetros, para verificar en Kaggle que todo funciona antes de lanzar la corrida completa.

In [ ]:
smoke_cfg = apply_overrides(cfg, ["cv.n_repeats=1", "training.max_epochs=3", "tuning.enabled=false"])
smoke = run_within_cohort(smoke_cfg, X, seg, OUT / "smoke", variant="ctnet", folds=[0], verbose=1)
metrics.binary_metrics(*metrics.aggregate(smoke)[["label", "prob"]].to_numpy().T)

## 5. Experimento I (y variantes del Experimento II): CV anidada, repetida y disjunta por sujeto (§2.4, §2.13)

- Una corrida por familia de tareas (análisis primario) y otra con todas las tareas (agregación balanceada por tarea).
- Variantes sobre los mismos segmentos log-Mel (`pipeline.VARIANTS`): `ctnet` (GAP), `cnn_only` (sin Transformer), `ctnet_flatten` (cabeza del estudio preliminar), `resnet50` y `efficientnetb0` (CNN preentrenadas en ImageNet, Exp. II).
- Las corridas son **reanudables**: si Kaggle se corta, vuelve a ejecutar la celda y se saltan los folds ya terminados (`runs.jsonl`).

In [ ]:
RUN_FULL = False
FAMILIES = ["ddk_pataka", "monologue", "vowels", "repeat", None]   # None = all tasks
SPECTROGRAM_VARIANTS = ["ctnet", "cnn_only", "ctnet_flatten", "resnet50", "efficientnetb0"]

def family_subset(X, seg, family):
    if family is None:
        return X, seg
    keep = seg.task_family.eq(family).to_numpy()
    return X[keep], seg[keep].reset_index(drop=True)

if RUN_FULL:
    for fam in FAMILIES:
        Xf, segf = family_subset(X, seg, fam)
        if len(segf) == 0:
            print("skip", fam); continue
        for variant in SPECTROGRAM_VARIANTS:  # ctnet first: its folds.csv is reused by every baseline
            run_within_cohort(variant_config(cfg, variant), Xf, segf, OUT / "neurovoz" / (fam or "all_tasks"),
                              variant=variant, save_models=(variant == "ctnet"))

## 6. Resultados por sujeto (§2.15)

In [ ]:
METRICS = ["auroc", "pr_auc", "balanced_accuracy", "sensitivity", "specificity", "f1", "brier", "cal_slope"]

def summarize(pred_path, task_balanced):
    subj = metrics.aggregate(pd.read_csv(pred_path), task_balanced=task_balanced)
    per_rep, summary = metrics.repeated_cv_summary(subj, cfg.evaluation.threshold)
    row = {}
    for m in METRICS:
        point, lo, hi = metrics.bootstrap_ci(subj, m, cfg.evaluation.threshold,
                                             n_boot=cfg.evaluation.n_bootstrap, level=cfg.evaluation.ci_level)
        row[m] = f"{point:.3f} [{lo:.3f}, {hi:.3f}]"
    row["auroc_sd_across_repeats"] = round(summary.loc["auroc", "std"], 3)
    return row, subj

results, subject_preds = [], {}
for fam_dir in sorted((OUT / "neurovoz").glob("*")):
    for var_dir in sorted(fam_dir.glob("*")):
        if (var_dir / "predictions.csv").exists():
            row, subj = summarize(var_dir / "predictions.csv", task_balanced=fam_dir.name == "all_tasks")
            results.append({"task_family": fam_dir.name, "variant": var_dir.name, **row})
            subject_preds[(fam_dir.name, var_dir.name)] = subj
results = pd.DataFrame(results)
results.to_csv(OUT / "exp1_results.csv", index=False)
results

In [ ]:
# Paired comparisons on identical subjects: CTNet vs every baseline (exploratory, Holm within each task family).
# DeLong uses repeat 0; all models share the same folds.csv, so the subjects are identical.
from ctnet_pd.stats import delong_test, holm
comparisons = []
for (fam, var), subj in subject_preds.items():
    if var == "ctnet" or (fam, "ctnet") not in subject_preds:
        continue
    a = subject_preds[(fam, "ctnet")].query("repeat == 0").set_index("subject_id")
    b = subj.query("repeat == 0").set_index("subject_id").loc[a.index]
    comparisons.append({"task_family": fam, "baseline": var,
                        **delong_test(a.label.to_numpy(), a.prob.to_numpy(), b.prob.to_numpy())})
comparisons = pd.DataFrame(comparisons)
if len(comparisons):
    comparisons["p_holm"] = comparisons.groupby("task_family")["p"].transform(lambda p: holm(p.to_numpy()))
comparisons

## 7. Mapas pre/post-atención y mapa conjunto (§2.8)

Se calculan sobre la rejilla nativa 14×25; la vista a 128×229 es solo para visualización.

In [ ]:
FAMILY, REP, FOLD = "ddk_pataka", 0, 0
run_dir = OUT / "neurovoz" / FAMILY / "ctnet"
Xf, segf = family_subset(X, seg, FAMILY)
model, norm_stats, test_subjects = analysis.load_fold(run_dir, REP, FOLD)
maps = analysis.fold_explanations(model, norm_stats, Xf, segf, test_subjects, target="pred")

def show_maps(i):
    fig, axes = plt.subplots(1, 4, figsize=(18, 3))
    axes[0].imshow(maps["X"][i], origin="lower", aspect="auto", cmap="magma"); axes[0].set_title("log-Mel (standardized)")
    for ax, key, title in zip(axes[1:], ["q_pre", "q_post", "q_joint"], ["pre-attention", "post-attention", "joint q_J"]):
        ax.imshow(xai.to_input_resolution(maps[key][i]), origin="lower", aspect="auto", cmap="viridis"); ax.set_title(title)
    s = maps["seg"].iloc[i]
    fig.suptitle(f"{s.subject_id} | label {s.label} | explained class {maps['target'][i]}"); plt.show()

for label in (0, 1):
    show_maps(int(np.flatnonzero(maps["seg"].label.to_numpy() == label)[0]))

In [ ]:
# Pre/post agreement (descriptive, never optimized; §2.11.6)
from scipy.stats import spearmanr
agree = pd.DataFrame({
    "spearman": [spearmanr(a.ravel(), b.ravel()).statistic for a, b in zip(maps["q_pre"], maps["q_post"])],
    **{f"jaccard_{q:.2f}": [sanity.topk_iou(a, b, q) for a, b in zip(maps["q_pre"], maps["q_post"])]
       for q in cfg.xai.agreement_q},
    "subject_id": maps["seg"].subject_id})
agree.groupby("subject_id").mean().describe().loc[["mean", "std"]]

In [ ]:
# Frequency profiles by class (time marginalized; §2.12.1)
from ctnet_pd.grid import mel_row_frequencies
hz = mel_row_frequencies(rows, cfg.spectrogram.n_mels, cfg.audio.sample_rate)
true_maps = analysis.fold_explanations(model, norm_stats, Xf, segf, test_subjects, target="true")
subjects_, subj_maps = reproducibility.subject_mean_maps(true_maps["q_joint"], true_maps["seg"].subject_id)
lab = true_maps["seg"].drop_duplicates("subject_id").set_index("subject_id").loc[subjects_, "label"].to_numpy()
prof = reproducibility.frequency_profiles(subj_maps)
fig, ax = plt.subplots(figsize=(7, 3.5))
for c, name in ((0, "HC"), (1, "PD")):
    ax.plot(hz, prof[lab == c].mean(axis=0), marker="o", label=f"{name} (n={int((lab == c).sum())})")
ax.set(xscale="log", xlabel="frequency (Hz, row centre)", ylabel="mean q_J", title=f"{FAMILY}, fold {FOLD}"); ax.legend(); plt.show()

## 8. Fidelidad: curvas de borrado/inserción contra máscaras aleatorias emparejadas (§2.11, H1)

Las entradas están estandarizadas por banda, así que el baseline gaussiano μ_f + σ_f·ε es ruido normal estándar.

In [ ]:
RUN_FAITH = False
if RUN_FAITH:
    faith = []
    for fold in range(cfg.cv.n_outer_folds):
        m_, st_, ts_ = analysis.load_fold(run_dir, REP, fold)
        mp_ = analysis.fold_explanations(m_, st_, Xf, segf, ts_)
        for key in ("q_pre", "q_post", "q_joint"):
            faith.append(analysis.fold_faithfulness(m_, mp_, cfg, map_key=key, max_segments_per_subject=3).assign(fold=fold))
    faith = pd.concat(faith, ignore_index=True)
    faith.to_csv(OUT / f"faithfulness_{FAMILY}.csv", index=False)
    display(faith.groupby("map")[["gap", "gap_random", "auc_del", "auc_ins"]].mean())
    display(pd.DataFrame({k: analysis.h1_test(g) for k, g in faith.groupby("map")}).T)

## 9. Pruebas de sanidad: aleatorización en cascada (§2.11.5)

In [ ]:
idx = np.arange(min(16, len(maps["X"])))
target = maps["target"][idx]
explain_fn = lambda key: (lambda m, Xs: xai.explain(m, Xs, target)[key])
cascade = pd.concat([pd.DataFrame(sanity.cascading_randomization(model, explain_fn(k), maps["X"][idx])).assign(map=k)
                     for k in ("q_pre", "q_post", "q_joint")])
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, g in cascade.groupby("map"):
    ax.plot(g.randomized_up_to, g.spearman_abs, marker="o", label=k)
ax.set(ylabel="|Spearman| vs trained", title="Cascading randomization (top -> bottom)"); ax.legend(); plt.show()
cascade

## 10. Experimento III y reproducibilidad entre cohortes (§2.12, §2.13)

Requiere PC-GITA. Configura `cohorts.pcgita.*` en el YAML (ruta y `filename_regex`) y ejecuta. Con una sola cohorte, aquí se calcula solo el techo de ruido (split-half) de NeuroVoz.

In [ ]:
ceiling_nv = reproducibility.split_half_ceiling(prof, lab, cfg.reproducibility.n_split_half)
print("NeuroVoz split-half ceiling (Spearman-Brown):", round(ceiling_nv["rho_sb_median"], 3))

HAVE_PCGITA = bool(cfg.cohorts.pcgita.root)
if HAVE_PCGITA:
    idx_g = data.index_cohort(cfg, "pcgita")
    Xg, segg = features.build_features(idx_g, cfg, cache_path=OUT / "cache" / "pcgita_features")
    Xg_f, segg_f = family_subset(Xg, segg, FAMILY)
    # Exp. III, both directions, for CTNet and the spectrogram baselines
    for variant in SPECTROGRAM_VARIANTS:
        run_external(variant_config(cfg, variant), Xg_f, segg_f, Xf, segf,
                     OUT / "external" / f"pcgita_to_neurovoz_{FAMILY}", variant=variant)
        run_external(variant_config(cfg, variant), Xf, segf, Xg_f, segg_f,
                     OUT / "external" / f"neurovoz_to_pcgita_{FAMILY}", variant=variant)
    # Feature / embedding baselines: baselines.run_feature_external and baselines.run_embedding_external
    # Scenario B: profiles of each cohort from its own out-of-fold models (repeat 0, all folds),
    # then: reproducibility.permutation_null(prof_g, lab_g, prof, lab, cfg.reproducibility.n_permutations)

## 11. Baselines del Experimento II que no usan log-Mel

Usan el mismo `folds.csv` de CTNet, de modo que todos los modelos se evalúan sobre los mismos sujetos. Sus predicciones quedan en `OUT/neurovoz/<familia>/<baseline>/predictions.csv`, así que las secciones 6 y la comparación pareada los incluyen al volver a ejecutarlas.

- **Rasgos por grabación + SVM:** eGeMAPSv02 (openSMILE) y medidas clásicas de Praat (jitter, shimmer, HNR, F0, formantes).
- **Encoders congelados + regresión logística:** WavLM-Base+, XLS-R 300M y AST (AudioSet). La capa se elige en la CV interna, y las grabaciones largas se procesan por trozos de 10 s en vez de truncarse.

openSMILE y PyTorch fallan si corren en el mismo proceso que TensorFlow, así que la extracción se hace en un subproceso aislado, de forma transparente. Los features y embeddings se guardan en `OUT/cache`.

In [ ]:
RUN_BASELINES = False
FEATURE_BASELINES = ["egemaps", "praat"]
EMBEDDING_BASELINES = ["wavlm", "xlsr", "ast"]   # keys of baselines.embedding_models in the YAML

if RUN_BASELINES:
    # !pip install -q opensmile praat-parselmouth transformers
    from ctnet_pd import baselines

    # Extract once for every recording (cached), then subset per task family.
    tables = {"egemaps": baselines.egemaps_features, "praat": baselines.praat_features}
    feature_tables = {name: tables[name](index, cfg, cache_path=OUT / "cache" / f"neurovoz_{name}")
                      for name in FEATURE_BASELINES}
    for name in EMBEDDING_BASELINES:
        baselines.pretrained_embeddings(index, cfg, name, cache_path=OUT / "cache" / f"neurovoz_{name}")

    def save(pred, fam_name, variant):
        d = OUT / "neurovoz" / fam_name / variant
        d.mkdir(parents=True, exist_ok=True)
        pred.to_csv(d / "predictions.csv", index=False)

    for fam in FAMILIES:
        fam_name = fam or "all_tasks"
        folds_path = OUT / "neurovoz" / fam_name / "ctnet" / "folds.csv"
        if not folds_path.exists():
            print("run CTNet first for", fam_name); continue
        assign = pd.read_csv(folds_path)
        idx_f = (index if fam is None else index[index.task_family == fam]).reset_index(drop=True)
        for name, table in feature_tables.items():
            X_b = baselines.feature_matrix(table, idx_f)
            save(baselines.run_feature_baseline(X_b, idx_f, assign, cfg, "svm", variant=f"{name}_svm"),
                 fam_name, f"{name}_svm")
        for name in EMBEDDING_BASELINES:
            E = baselines.pretrained_embeddings(idx_f, cfg, name, cache_path=OUT / "cache" / f"neurovoz_{name}")
            save(baselines.run_embedding_probe(E, idx_f, assign, cfg, variant=f"{name}_probe"),
                 fam_name, f"{name}_probe")
        print("baselines done:", fam_name)

## 12. Registro del entorno (para la sección de software del artículo)

In [ ]:
print(json.dumps(VERSIONS, indent=2))